# Agent Platform 上での Kubeflow Pipeline の CI/CD

**学習目標:**
1. Agent Platform Pipelines を制御するためのカスタム Cloud Build ビルダーを作成する方法を学ぶ
1. KFP 向けのすべてのアーティファクトをビルドしてプッシュするための Cloud Build 構成ファイルを作成する方法を学ぶ
1. Kubeflow Pipeline の新しい実行をトリガーする Cloud Build GitHub トリガーを設定する方法を学ぶ

このラボでは、Agent Platform 上で Kubeflow Pipeline を自動的にビルド、デプロイ、実行する **Cloud Build** CI/CD ワークフローの作成手順を学習します。また、パイプラインのコードをホストする **GitHub** リポジトリに新しいタグが適用されたときにワークフローを開始するトリガーを設定し、ワークフローを **GitHub** と統合します。

In [ ]:
import os

## 環境設定の構成

In [ ]:
PROJECT_ID = !(gcloud config get-value project)
PROJECT_ID = PROJECT_ID[0]
REGION = "us-central1"
ARTIFACT_STORE = f"gs://{PROJECT_ID}-kfp-artifact-store"
os.environ["REGION"] = REGION
os.environ["ARTIFACT_STORE"] = ARTIFACT_STORE

アーティファクト ストアが存在することを確認します。

In [ ]:
!gcloud storage ls | grep ^{ARTIFACT_STORE}/$ || gcloud storage buckets create --location={REGION} {ARTIFACT_STORE}

また、このノートブックでは、[ウォークスルー ノートブック](https://github.com/GoogleCloudPlatform/asl-ml-immersion/blob/master/notebooks/kubeflow_pipelines/walkthrough/solutions/kfp_walkthrough_vertex.ipynb) で説明されている手順に従って、データセットがすでに作成され Google Cloud Storage に保存されていることを前提としています。

まだ実行していない場合は、パイプラインを実行する前に下のセルを実行してデータセットを作成してください。

In [ ]:
%%bash
gcloud storage cp gs://asl-public-data/data/covertype/training/dataset.csv $ARTIFACT_STORE/data/training/dataset.csv
gcloud storage cp gs://asl-public-data/data/covertype/validation/dataset.csv $ARTIFACT_STORE/data/validation/dataset.csv

## Agent Platform 向け KFP CLI ビルダーの作成
### KFP CLI ビルダーを記述した Dockerfile の確認

In [ ]:
!cat kfp-cli/Dockerfile

### イメージをビルドしてプロジェクトの **Artifact Registry** にプッシュする

In [ ]:
ARTIFACT_REGISTRY_DIR = "asl-artifact-repo"
KFP_CLI_IMAGE_NAME = "kfp-cli"
KFP_CLI_IMAGE_URI = f"us-docker.pkg.dev/{PROJECT_ID}/{ARTIFACT_REGISTRY_DIR}/{KFP_CLI_IMAGE_NAME}:latest"
KFP_CLI_IMAGE_URI

In [ ]:
!gcloud builds submit --timeout 15m --tag {KFP_CLI_IMAGE_URI} kfp-cli

## **Cloud Build** ワークフローの理解

`cloudbuild.yaml` ファイルを確認して、CI/CD ワークフローがどのように実装されているか、および環境固有の設定が **Cloud Build** 変数を使用してどのように抽象化されているかを理解します。

この CI/CD ワークフローは、[kfp_pipeline_lightweight.ipynb](../../pipelines/solutions/kfp_pipeline_lightweight.ipynb) ノートブックで手動で行った手順を自動化します。
1. トレーナー イメージのビルド
1. パイプラインのコンパイル
1. パイプラインを Agent Platform Pipeline 環境にアップロードして実行
1. トレーナーをプロジェクトの **Artifact Registry** にプッシュ
 

**Cloud Build** ワークフロー構成では、標準およびカスタムの [Cloud Build ビルダー](https://cloud.google.com/cloud-build/docs/cloud-builders) の両方が使用されます。カスタム ビルダーは **KFP CLI** をカプセル化しています。

ビルドログを保存するための GCS バケットを作成しましょう。

In [ ]:
BUCKET = PROJECT_ID + "-cicd-log"
os.environ["BUCKET"] = BUCKET

In [ ]:
%%bash

exists=$(gcloud storage ls --buckets | grep -w gs://${BUCKET}/)
if [ -n "$exists" ]; then
    echo -e "Bucket exists, let's not recreate it."
else
    echo "Creating a new GCS bucket."
    gcloud storage buckets create --location=${REGION} gs://${BUCKET}
    echo "Here are your current buckets:"
    gcloud storage ls
fi

## CI/CD 実行の手動トリガー

[gcloud builds submit コマンド](https://cloud.google.com/sdk/gcloud/reference/builds/submit) を使用して、**Cloud Build** の実行を手動でトリガーできます。

In [ ]:
SUBSTITUTIONS = f"_REGION={REGION},_PIPELINE_FOLDER=./"
SUBSTITUTIONS

In [ ]:
!gcloud builds submit . --config cloudbuild.yaml --substitutions {SUBSTITUTIONS}

**注:** Cloud Build が Agent Platform にアクセスする際に問題が発生した場合は、**Cloud Shell** で次のコマンドを実行する必要がある場合があります。

```
PROJECT_ID=$(gcloud config get-value project)
PROJECT_NUMBER=$(gcloud projects list --filter="name=$PROJECT_ID" --format="value(PROJECT_NUMBER)")

gcloud projects add-iam-policy-binding $PROJECT_ID \
  --member serviceAccount:$PROJECT_NUMBER@cloudbuild.gserviceaccount.com \
  --role roles/editor
gcloud projects add-iam-policy-binding $PROJECT_ID \
    --member serviceAccount:$PROJECT_NUMBER-compute@developer.gserviceaccount.com \
    --role roles/storage.objectAdmin
```

## GitHub 統合の設定

この演習では、[Cloud Build GitHub アプリ](https://github.com/marketplace/google-cloud-build) を使用して、CI/CD ワークフローを **GitHub** と統合します。
パイプラインのソースコードを管理する **GitHub** リポジトリに新しいタグが適用されたときに CI/CD ワークフローを開始するトリガーを設定します。ソース GitHub リポジトリとして、このリポジトリのフォークを使用します。

### このリポジトリのフォークの作成
[GitHub のドキュメントに従って](https://help.github.com/en/github/getting-started-with-github/fork-a-repo)、[このリポジトリ](https://github.com/GoogleCloudPlatform/asl-ml-immersion) をフォークします。

### **Cloud Build** トリガーの作成

前の手順で作成したフォークを Google Cloud プロジェクトに接続し、[GitHub アプリのトリガーの作成](https://cloud.google.com/cloud-build/docs/create-github-app-triggers) の記事の手順に従ってトリガーを作成します。**[トリガーの編集]** フォームで次の値を設定します。

|項目| 値 |
|-----|--------------------------------------------------------------------------------|
|名前| [YOUR TRIGGER NAME] |
|説明| [YOUR TRIGGER DESCRIPTION] |
|イベント| タグ |
|ソース| [YOUR FORK] |
|タグ（正規表現）| .\* |
|ビルド構成| Cloud Build 構成ファイル（yaml または json） |
|Cloud Build 構成ファイルの場所| ./asl_mlops/notebooks/kubeflow_pipelines/cicd/solutions/cloudbuild.yaml |
|サービス アカウント| `<PROJECT NUMBER>-compute@developer.gserviceaccount.com` |


代入変数（Substitution variables）には次の値を使用します。

|変数|値|
|--------|-----|
|_REGION|us-central1|
|_PIPELINE_FOLDER|asl_mlops/notebooks/kubeflow_pipelines/cicd/solutions

### ビルドのトリガー

自動ビルドを開始するには、[GitHub でリポジトリの新しいリリースを作成](https://help.github.com/en/github/administering-a-repository/creating-releases) します。または、`git` を使用してタグを適用することでビルドを開始することもできます。
```
git tag [TAG NAME]
git push origin --tags
```


上記のコマンドを実行すると、ビルドが自動的にトリガーされ、[ここ](https://console.cloud.google.com/cloud-build/builds) で確認できるようになります。

Copyright 2021 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.